In [51]:
import pandas as pd
import torch

In [52]:
df = pd.read_csv("IMT2024007_train_var2.csv")
tdf = pd.read_csv("IMT2024007_test_var2.csv")

In [53]:
x = df.drop("y",axis = 1).values
y = df["y"].values

In [54]:
x = torch.tensor(x,dtype = torch.float32)
y = torch.tensor(y,dtype = torch.float32)

In [55]:
num_samples = x.shape[0]
indices = torch.randperm(num_samples)

train_size = int(0.9*num_samples)
train_idx = indices[ :train_size]
val_idx = indices[train_size : ]

x_train = x[train_idx]
y_train = y[train_idx]

x_val = x[val_idx]
y_val = y[val_idx]

In [56]:
def polynomial(x,n):
    samples,features = x.shape
    bias_col = torch.ones(samples,dtype = x.dtype)
    poly_terms = [bias_col]
    curr_level = []
    #Degree 1
    for i in range (features):
        col = x[ : , i]
        poly_terms.append(col)
        curr_level.append((col,i))

    for degree in range(2,n+1):
        new_level = []
        for term,indx in curr_level:
            for j in range (indx,features):
                new_term = term * x[ : , j]
                poly_terms.append(new_term)
                new_level.append((new_term,j))
            
        curr_level = new_level
    return torch.stack(poly_terms,dim = 1)

In [57]:
# Lists to store the error values for plotting later
train_mse_list = []
val_mse_list = []
val_r2_list = []
degrees = list(range(1, 21))

y_train_mean = torch.mean(y_train)
train_total_variance = torch.sum((y_train - y_train_mean) ** 2)

y_val_mean = torch.mean(y_val)
val_total_variance = torch.sum((y_val - y_val_mean) ** 2)

for d in degrees:
    # Expanding the features for both datasets
    A_train = polynomial(x_train, d)
    A_val = polynomial(x_val, d)
    
    # Calculating best weights using pseudo-inverse as discussed in class
    w = torch.pinverse(A_train) @ y_train
    
    # predictions
    y_train_pred = A_train @ w
    y_val_pred = A_val @ w
    
    # calculations of metrics(MSE,R^2)
    train_mse = torch.mean((y_train - y_train_pred)**2).item()
    val_mse = torch.mean((y_val - y_val_pred)**2).item()

    
    train_model_error = torch.sum((y_train - y_train_pred) ** 2)
    train_r2 = 1 - (train_model_error / train_total_variance).item()
    
    val_model_error = torch.sum((y_val - y_val_pred) ** 2)
    val_r2 = 1 - (val_model_error / val_total_variance).item()
    
    # 5. Save the results
    train_mse_list.append(train_mse)
    val_mse_list.append(val_mse)
    val_r2_list.append(val_r2)
    # Print a clean summary for each degree
    print(f"Degree {d:2d} | Train MSE: {train_mse:10.4f} (R2: {train_r2:7.4f}) | Val MSE: {val_mse:10.4f} (R2: {val_r2:7.4f})")

Degree  1 | Train MSE:    36.9774 (R2:  0.2369) | Val MSE:    21.0785 (R2:  0.1689)
Degree  2 | Train MSE:    20.9574 (R2:  0.5675) | Val MSE:    14.1979 (R2:  0.4402)
Degree  3 | Train MSE:    11.5503 (R2:  0.7616) | Val MSE:     6.3640 (R2:  0.7491)
Degree  4 | Train MSE:     3.2923 (R2:  0.9321) | Val MSE:     2.5215 (R2:  0.9006)
Degree  5 | Train MSE:     1.3281 (R2:  0.9726) | Val MSE:     1.1224 (R2:  0.9557)
Degree  6 | Train MSE:     0.4504 (R2:  0.9907) | Val MSE:     0.5313 (R2:  0.9791)
Degree  7 | Train MSE:     0.2446 (R2:  0.9950) | Val MSE:     0.3288 (R2:  0.9870)
Degree  8 | Train MSE:     0.1732 (R2:  0.9964) | Val MSE:     0.3273 (R2:  0.9871)
Degree  9 | Train MSE:     0.1527 (R2:  0.9968) | Val MSE:     0.3155 (R2:  0.9876)
Degree 10 | Train MSE:     0.1347 (R2:  0.9972) | Val MSE:     0.4980 (R2:  0.9804)
Degree 11 | Train MSE:     0.1207 (R2:  0.9975) | Val MSE:     0.4509 (R2:  0.9822)
Degree 12 | Train MSE:     0.0991 (R2:  0.9980) | Val MSE:     1.0310 (R2:  

In [58]:
"""after a few iterations it's clearly evident that the degree 8 outperforms all models with an insanely 
   good validation mse and R^2 , so i'll be choosing degree 8 polynomial, we can clearly see the model starts
   overfitting after degree 12 models, so the more the degree the more complex the weights are leading
   to memorising the training dataset"""

"after a few iterations it's clearly evident that the degree 8 outperforms all models with an insanely \n   good validation mse and R^2 , so i'll be choosing degree 8 polynomial, we can clearly see the model starts\n   overfitting after degree 12 models, so the more the degree the more complex the weights are leading\n   to memorising the training dataset"

In [59]:
A_train = polynomial(x,8)
w = torch.pinverse(A_train) @ y

In [60]:
x_test = tdf.values
x_test = torch.tensor(x_test ,dtype = torch.float32)

A_test = polynomial(x_test,8)
y_test = A_test @ w

In [61]:
pred_df = pd.DataFrame(y_test.numpy(),columns = ["y"])
pred_df.to_csv("IMT2024007_pred_var_2.csv",index = False)